In [1]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [2]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)

df = housing.frame
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [3]:
from sklearn.model_selection import train_test_split

data = df.values  # convert to numpy
X_train, X_test, y_train, y_test = train_test_split(data[:,:-1], data[:,-1], test_size=0.2, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.25, random_state=42)


In [4]:
X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

y_train = torch.FloatTensor(y_train).reshape(-1,1)
y_valid = torch.FloatTensor(y_valid).reshape(-1,1)
y_test = torch.FloatTensor(y_test).reshape(-1,1)

In [5]:
torch.manual_seed(42)
n_features = X_train.shape[1]
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

## low level implementation

In [6]:
learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_train - y_pred)**2).mean()
    loss.backward()
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f"Epoch {epoch + 1}/{n_epochs}, loss:{loss.item()}")

Epoch 1/20, loss:16.006189346313477
Epoch 2/20, loss:4.656662940979004
Epoch 3/20, loss:2.104856491088867
Epoch 4/20, loss:1.2392686605453491
Epoch 5/20, loss:0.9124189615249634
Epoch 6/20, loss:0.7779617309570312
Epoch 7/20, loss:0.7152504920959473
Epoch 8/20, loss:0.6805714964866638
Epoch 9/20, loss:0.6576952934265137
Epoch 10/20, loss:0.6404299736022949
Epoch 11/20, loss:0.6263093948364258
Epoch 12/20, loss:0.6142740249633789
Epoch 13/20, loss:0.6038088798522949
Epoch 14/20, loss:0.5946188569068909
Epoch 15/20, loss:0.5865059494972229
Epoch 16/20, loss:0.5793203711509705
Epoch 17/20, loss:0.5729405879974365
Epoch 18/20, loss:0.5672646760940552
Epoch 19/20, loss:0.5622055530548096
Epoch 20/20, loss:0.5576879382133484


In [7]:
import torch.nn as nn

torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1)

In [8]:
print(model.bias, model.weight)

Parameter containing:
tensor([0.3117], requires_grad=True) Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)


In [9]:
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)
mse = nn.MSELoss()

In [10]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

In [11]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 4.272577285766602
Epoch 2/20, Loss: 0.7673617601394653
Epoch 3/20, Loss: 0.6151816248893738
Epoch 4/20, Loss: 0.5950986742973328
Epoch 5/20, Loss: 0.5839645862579346
Epoch 6/20, Loss: 0.5752211213111877
Epoch 7/20, Loss: 0.567899227142334
Epoch 8/20, Loss: 0.5616247057914734
Epoch 9/20, Loss: 0.5561908483505249
Epoch 10/20, Loss: 0.5514596104621887
Epoch 11/20, Loss: 0.5473266839981079
Epoch 12/20, Loss: 0.5437079668045044
Epoch 13/20, Loss: 0.5405330061912537
Epoch 14/20, Loss: 0.5377423763275146
Epoch 15/20, Loss: 0.5352851748466492
Epoch 16/20, Loss: 0.5331176519393921
Epoch 17/20, Loss: 0.5312021970748901
Epoch 18/20, Loss: 0.529506504535675
Epoch 19/20, Loss: 0.5280026197433472
Epoch 20/20, Loss: 0.5266665816307068


In [12]:
torch.manual_seed(42)
n_features = X_train.shape[1]
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)
print(n_features)

8


In [13]:
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50,40),
    nn.ReLU(),
    nn.Linear(40,1)
)

In [14]:
learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 6.2036213874816895
Epoch 2/20, Loss: 2.8577897548675537
Epoch 3/20, Loss: 1.3390477895736694
Epoch 4/20, Loss: 1.0098811388015747
Epoch 5/20, Loss: 0.8722575306892395
Epoch 6/20, Loss: 0.7905676960945129
Epoch 7/20, Loss: 0.7395867109298706
Epoch 8/20, Loss: 0.707360029220581
Epoch 9/20, Loss: 0.6854878664016724
Epoch 10/20, Loss: 0.6693881750106812
Epoch 11/20, Loss: 0.6565759181976318
Epoch 12/20, Loss: 0.645799994468689
Epoch 13/20, Loss: 0.6363627314567566
Epoch 14/20, Loss: 0.6279155611991882
Epoch 15/20, Loss: 0.6202101111412048
Epoch 16/20, Loss: 0.6131242513656616
Epoch 17/20, Loss: 0.6063646674156189
Epoch 18/20, Loss: 0.599853515625
Epoch 19/20, Loss: 0.5935519933700562
Epoch 20/20, Loss: 0.5874618887901306


In [15]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True, num_workers=8, persistent_workers=True)

In [16]:
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40,1)
)

model.to(device)

Sequential(
  (0): Linear(in_features=8, out_features=50, bias=True)
  (1): ReLU()
  (2): Linear(in_features=50, out_features=40, bias=True)
  (3): ReLU()
  (4): Linear(in_features=40, out_features=1, bias=True)
)

In [21]:
def train(model, optimizer, criterion, train_loader:DataLoader, n_epochs:int):
    model.train()
    optimizer.zero_grad()
    for epoch in range(n_epochs):
        total_loss = 0.
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device, non_blocking=True), y_batch.to(device, non_blocking=True)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        
        mean_loss = total_loss/len(train_loader)
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {mean_loss:.4f}")

In [18]:
learning_rate = 0.0001
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train(model, optimizer, mse, train_loader, n_epochs)

Epoch 1/20, Loss: 4.7305
Epoch 2/20, Loss: 3.7946
Epoch 3/20, Loss: 3.0408
Epoch 4/20, Loss: 2.4295
Epoch 5/20, Loss: 1.9582
Epoch 6/20, Loss: 1.6177
Epoch 7/20, Loss: 1.3827
Epoch 8/20, Loss: 1.2209
Epoch 9/20, Loss: 1.1062
Epoch 10/20, Loss: 1.0190
Epoch 11/20, Loss: 0.9485
Epoch 12/20, Loss: 0.8892
Epoch 13/20, Loss: 0.8391
Epoch 14/20, Loss: 0.7964
Epoch 15/20, Loss: 0.7595
Epoch 16/20, Loss: 0.7282
Epoch 17/20, Loss: 0.7016
Epoch 18/20, Loss: 0.6787
Epoch 19/20, Loss: 0.6592
Epoch 20/20, Loss: 0.6424


## Model Evaluation

In [22]:
def evaluate(model, data_loader, metric_fn, aggregate_fn = torch.mean):
    model.eval()
    metrics = []
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric = metric_fn(y_pred, y_batch)
            metrics.append(metric)
    return aggregate_fn(torch.stack(metrics))

def rmse(y_pred, y_true):
    return ((y_pred - y_true) ** 2).mean().sqrt()

In [24]:
valid_dataset = TensorDataset(X_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=32)
valid_mse = evaluate(model, valid_loader, mse)
valid_mse

tensor(0.6806, device='cuda:0')

In [25]:
valid_mse.sqrt()

tensor(0.8250, device='cuda:0')

In [26]:
evaluate(model, valid_loader, mse, aggregate_fn=lambda metrics: torch.sqrt(torch.mean(metrics)))

tensor(0.8250, device='cuda:0')

In [27]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

In [28]:
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
evaluate_tm(model, valid_loader, rmse)

tensor(0.8250, device='cuda:0')